# 01 — HDSD preprocessing and baseline

Read the original recordings and annotations, reproduce the fixed manifest, and measure pretrained Whisper before adaptation.

Use **HDSD Research** and run from top to bottom. Every workflow is written here; there are no project-helper imports. The few local functions below perform small I/O, scoring, or padding operations and show their complete code. M2 only, supplied labels accepted, no augmentation. Previous results stay intact.


## 1. Locate the project and check the GPU

A path tells Python where a file lives. We pin the exact pretrained model revision and Hindi generation settings so every comparison has the same starting point. Nothing is trained in this cell.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import csv, gc, hashlib, importlib.metadata, json, math, platform, re, statistics, time, unicodedata
import numpy as np
import soundfile as sf
import torch
from torch.utils.data import DataLoader
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from huggingface_hub import snapshot_download
from huggingface_hub.errors import LocalEntryNotFoundError
from IPython.display import Audio, display

ROOT = next(path for path in [Path.cwd(), Path.cwd()/"hdsd_simple", Path("/workspace/hdsd_simple")]
            if (path/"HDSD").is_dir() and (path/"01_HDSD_Preprocessing_Baseline.ipynb").is_file())
CORPUS = ROOT/"HDSD"
MANIFEST = ROOT/"prepared/v1/manifest.jsonl"
PAPER = ROOT/"runs/paper"
MODEL = "openai/whisper-small"
REVISION = "973afd24965f72e36ca33b3055d56a652f456b4d"
GENERATION = {"language": "hi", "task": "transcribe", "return_timestamps": False,
              "do_sample": False, "num_beams": 1, "max_new_tokens": 96}
NORMALIZATION = "nfc_unicode_punctuation_to_space_whitespace_v1"
REFERENCE_SOURCE = "supplied Roman WORD annotations + documented Hindi conversion"
LEGACY_HELPER_SHA256 = "a99510cdc63a1fba97b98455cc4a8b47f941205f94df92d0d55d0081bb08af8e"
LEGACY_RESEARCH_SHA256 = "7f6cfb9e9f637a1364476fcdf3849608b15335fec10f74db50d17a199eeba855"
PREDICTION_FIELDS = ["utterance_id", "speaker", "prompt", "reference", "prediction",
                     "reference_status", "audio_sha256", "generation_cap_reached"]
GROUPS = {"unseen_speakers_familiar_sentences": "dev_speaker_train_prompt",
          "familiar_speakers_unseen_sentences": "train_speaker_dev_prompt",
          "unseen_speakers_unseen_sentences": "dev"}
assert torch.cuda.is_available(), "Select the HDSD Research GPU kernel."
assert torch.cuda.is_bf16_supported(), "These training settings require BF16."
ENVIRONMENT = {"python": platform.python_version(), "torch": torch.__version__,
               "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(),
               "allocated_gpu_bytes": torch.cuda.get_device_properties(0).total_memory,
               "packages": {name: importlib.metadata.version(name) for name in
                  ["transformers", "peft", "soundfile", "numpy", "accelerate", "librosa"]}}
print("Project:", ROOT)
print(json.dumps(ENVIRONMENT, indent=2))

NOTEBOOK_NAME = '01_HDSD_Preprocessing_Baseline.ipynb'
NOTEBOOK_FILES = ['01_HDSD_Preprocessing_Baseline.ipynb', '02_HDSD_Full_FineTuning.ipynb', '03_HDSD_PEFT_FineTuning.ipynb']


## 2. Read and write experiment files

These small functions show ordinary file operations. A SHA256 hash is a fingerprint: changing the bytes changes the fingerprint. CSV holds tabular predictions; JSON stores settings. Raw HDSD files are read only.


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def read_csv(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

def write_csv(path, rows, fields):
    with Path(path).open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

def corpus_path(root, relative):
    root = Path(root).resolve()
    path = (root / relative).resolve()
    if not path.is_relative_to(root):
        raise ValueError(f"Path escapes corpus: {relative}")
    if not path.is_file():
        raise FileNotFoundError(path)
    return path


## 3. Record the code identity

We fingerprint Python operations rather than cell IDs or formatting. Editable run controls and the fingerprint-calculation block are excluded; their settings are recorded separately. Audited previous results retain their original fingerprints and are reused only while the scientific operations match the verified implementation.


In [ ]:
# Fingerprint Python operations, independent of cell IDs, formatting, and run controls.
import ast

def syntax_signature(node):
    # Ignore the empty generic-parameter field added in Python 3.12.
    if isinstance(node, ast.AST):
        fields = {field:syntax_signature(value) for field,value in ast.iter_fields(node)
                  if not (field == "type_params" and not value)}
        return [type(node).__name__, fields]
    if isinstance(node, list):
        return [syntax_signature(value) for value in node]
    if isinstance(node, bytes):
        return ["bytes", node.hex()]
    return node

CONTROL_NAMES = {"EXPERIMENT", "CONFIG", "RUN_NAME", "LEARNING_RATES", "SEEDS"}
NOTEBOOK_CODE_HASHES = {}
for name in NOTEBOOK_FILES:
    notebook = json.loads((ROOT/name).read_text())
    operations = []
    for cell in notebook["cells"]:
        if cell["cell_type"] != "code":
            continue
        tree = ast.parse("".join(cell["source"]))
        assigned = {target.id for node in tree.body if isinstance(node, ast.Assign)
                    for target in node.targets if isinstance(target, ast.Name)}
        is_controls = assigned == CONTROL_NAMES and all(isinstance(node, ast.Assign) for node in tree.body)
        is_fingerprint = {"NOTEBOOK_CODE_HASHES", "SOURCE_HASH"} <= assigned
        # Appended evaluation has a separately frozen code identity; training stays unchanged.
        is_final_evaluation = bool(tree.body and isinstance(tree.body[0], ast.Expr)
            and isinstance(tree.body[0].value, ast.Constant)
            and isinstance(tree.body[0].value.value, str)
            and tree.body[0].value.value.startswith("HDSD final evaluation v"))
        if is_controls or is_fingerprint or is_final_evaluation:
            continue
        operations.extend(syntax_signature(node) for node in tree.body)
    NOTEBOOK_CODE_HASHES[name] = hashlib.sha256(json.dumps(operations, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
SOURCE_HASH = NOTEBOOK_CODE_HASHES[NOTEBOOK_NAME]

# These old fingerprints were verified against the preserved executed notebooks.
# Accept them only while the scientific operations match this audited version.
AUDITED_ENGINE_HASHES = {'01_HDSD_Preprocessing_Baseline.ipynb': '24adb6d0773eb806c8077cd5c49dea8b288d0abc3b48c5c552b27ac899a0b6ba', '02_HDSD_Full_FineTuning.ipynb': 'a6db0bdcff15e23e15f62605534cd3bbd523172d810d8c11330777b4968be274', '03_HDSD_PEFT_FineTuning.ipynb': '18eaf7a10b2ddcd0f3c5bff5b9ca4f219e93ee8c2df06da36f1262ff6d35244a'}
AUDITED_PRIOR_HASHES = {'01_HDSD_Preprocessing_Baseline.ipynb': [], '02_HDSD_Full_FineTuning.ipynb': ['03775459333764c3df9a1435886e584e60d2e29de742f0e167812603d2db525e'], '03_HDSD_PEFT_FineTuning.ipynb': ['7849f2fc9e91aff2f7292faae3ba975fe1d4dde83e1c76ae65eeced10a7b2ec5']}
COMPATIBLE_CODE_HASHES = {name:{value} for name,value in NOTEBOOK_CODE_HASHES.items()}
for name,prior_hashes in AUDITED_PRIOR_HASHES.items():
    if NOTEBOOK_CODE_HASHES[name] == AUDITED_ENGINE_HASHES[name]:
        COMPATIBLE_CODE_HASHES[name].update(prior_hashes)
print("Scientific-code fingerprint (Python syntax v1):", SOURCE_HASH)


## 4. Define the fixed labels and partitions

This dictionary is the documented Roman-word to Hindi conversion used by the completed pilots. The three contextual corrections appear explicitly in the next cell. Repetitions and omissions are retained. Speaker/prompt lists reproduce the existing partition rather than randomly changing it.


In [ ]:
PARTITIONS = {
  "train": {
    "speakers": [
      "F11",
      "F12",
      "F29",
      "F34",
      "F35",
      "F45",
      "F57",
      "F59",
      "F60",
      "M01",
      "M03",
      "M05",
      "M08",
      "M09",
      "M10",
      "M13",
      "M15",
      "M16",
      "M18",
      "M21",
      "M22",
      "M24",
      "M25",
      "M28",
      "M32",
      "M33",
      "M36",
      "M37",
      "M38",
      "M39",
      "M40",
      "M41",
      "M43",
      "M46",
      "M48",
      "M49",
      "M50",
      "M53",
      "M55",
      "M56",
      "M62"
    ],
    "prompts": [
      "H01",
      "H02",
      "H03",
      "H04",
      "H05",
      "H06",
      "H07",
      "H08",
      "H11",
      "H12",
      "H14",
      "H15",
      "H17",
      "H19",
      "H20",
      "H21",
      "H22",
      "H23",
      "H26",
      "H27",
      "H30"
    ]
  },
  "dev": {
    "speakers": [
      "F19",
      "F31",
      "F51",
      "M14",
      "M27",
      "M30",
      "M42",
      "M47",
      "M58"
    ],
    "prompts": [
      "H09",
      "H18",
      "H24",
      "H29"
    ]
  },
  "test": {
    "speakers": [
      "F07",
      "F17",
      "F20",
      "F26",
      "F54",
      "F63",
      "M23",
      "M44",
      "M52"
    ],
    "prompts": [
      "H10",
      "H13",
      "H16",
      "H25",
      "H28"
    ]
  }
}

DRAFT_MAP = {
  "aapakei": "आपके",
  "karanei": "करने",
  "isakei": "इसके",
  "badalei": "बदले",
  "mein": "में",
  "isei": "इसे",
  "aisei": "ऐसे",
  "rakhein": "रखें",
  "kei": "के",
  "liyei": "लिए",
  "aagei": "आगे",
  "jaanei": "जाने",
  "kahatei": "कहते",
  "diijiyei": "दीजिए",
  "kiijiei": "कीजिए",
  "jaaei": "जाए",
  "rahei": "रहे",
  "padhtei": "पढ़ते",
  "hootaa": "होता",
  "karataa": "करता",
  "jaataa": "जाता",
  "rahataa": "रहता",
  "chalaa": "चला",
  "gayaa": "गया",
  "huaa": "हुआ",
  "thaa": "था",
  "liyaa": "लिया",
  "kahaa": "कहा",
  "deikhaa": "देखा",
  "bhuula": "भूला",
  "sakaa": "सका",
  "lagaa": "लगा",
  "aabhaara": "आभार",
  "dhanyavaada": "धन्यवाद",
  "svaagata": "स्वागत",
  "bhaarata": "भारत",
  "hindii": "हिंदी",
  "khushii": "खुशी",
  "jaldii": "जल्दी",
  "meirii": "मेरी",
  "abhii": "अभी",
  "bhii": "भी",
  "hii": "ही",
  "nahiin": "नहीं",
  "huiin": "हुईं",
  "jiivana": "जीवन",
  "krikeita": "क्रिकेट",
  "sandeisha": "संदेश",
  "sangharshha": "संघर्ष",
  "sahayooga": "सहयोग",
  "sahaaraa": "सहारा",
  "sanbhava": "संभव",
  "chikitsaa": "चिकित्सा",
  "unhoonnei": "उन्होंने",
  "looga": "लोग",
  "shaayada": "शायद",
  "shaanta": "शांत",
  "shuruu": "शुरू",
  "dhyaana": "ध्यान",
  "paalana": "पालन",
  "samajha": "समझ",
  "paakara": "पाकर",
  "maanaga": "मांग",
  "huuna": "हूं",
  "oora": "और",
  "aura": "और",
  "huii": "हुई",
  "jaarii": "जारी",
  "ilaaja": "इलाज",
  "kabhii": "कभी",
  "sabhii": "सभी",
  "hai": "है",
  "main": "मैं",
  "bahuta": "बहुत",
  "sei": "से",
  "kaa": "का",
  "naa": "ना",
  "hoo": "हो",
  "aisaa": "ऐसा",
  "isa": "इस",
  "joo": "जो",
  "pasanda": "पसंद",
  "ki": "कि",
  "kuchha": "कुछ",
  "aapa": "आप",
  "achchhaa": "अच्छा",
  "para": "पर",
  "artha": "अर्थ",
  "koo": "को",
  "yaha": "यह",
  "baatoon": "बातों",
  "madada": "मदद",
  "jhuutha": "झूठ",
  "kahataa": "कहता",
  "yahaan": "यहाँ",
  "tuma": "तुम",
  "heitu": "हेतु",
  "daura": "दौर",
  "doonoon": "दोनों",
  "antara": "अंतर",
  "kara": "कर",
  "aa": "आ",
  "dara": "दर",
  "usakaa": "उसका",
  "karoo": "करो",
  "usasei": "उससे",
  "kitanei": "कितने",
  "badaa": "बड़ा",
  "kheila": "खेल",
  "aapanei": "आपने",
  "phira": "फिर",
  "kaarya": "कार्य",
  "loo": "लो",
  "aba": "अब",
  "samaya": "समय",
  "saatha": "साथ",
  "bhaya": "भय",
  "baara": "बार",
  "kaaphii": "काफी",
  "baata": "बात",
  "vaha": "वह",
  "eika": "एक",
  "apanaa": "अपना",
  "aapakaa": "आपका",
  "magara": "मगर",
  "par": "पर",
  "pasand": "पसंद",
  "samayaa": "समय",
  "maang": "मांग",
  "hindi": "हिंदी"
}


## 5. Read recordings and prepare references

Each M2 filename gives speaker, session, and prompt. We read its WORD annotation, convert the spoken tokens, check both microphones, and record paths and hashes. M61 requires segmentation; four empty labels are excluded. Signal flags are recorded without automatically removing difficult examples. If the old manifest exists, the newly constructed version must match exactly.


In [ ]:
corpus = CORPUS
candidate_records, exclusions = [], []
for path in sorted((corpus/"hindi_sent").rglob("*.wav")):
    match = re.fullmatch(r"([FM]\d+)_(S\d+)_(H\d+)_M2",path.stem)
    if not match:
        if path.parent.name.startswith("M61"):
            exclusions.append({"utterance_id":"M61","path":str(path.relative_to(corpus)),
                               "reason":"irregular_stereo_recording_needs_segmentation"})
        continue
    speaker,session,prompt = match.groups()
    utterance_id = f"{speaker}_{session}_{prompt}"
    label = corpus/"WORD"/speaker/(utterance_id+".txt")
    roman = " ".join(line.split("\t")[2].strip() for line in label.read_text().splitlines() if line.strip())
    if not roman:
        exclusions.append({"utterance_id":utterance_id,"path":str(path.relative_to(corpus)),
                           "reason":"empty_or_irregular_label"})
        continue
    # Convert each spoken word. Do not replace utterances with the intended full prompt.
    hindi_words = []
    for token in roman.split():
        token = token.lower()
        assert token in DRAFT_MAP, f"Unmapped label token: {token}"
        word = DRAFT_MAP[token]
        if token == "oora": word = "ओर"
        if token == "ki" and prompt == "H10": word = "की"
        if token == "dara" and prompt == "H21": word = "डर"
        hindi_words.append(word)
    throat_path = path.with_name(utterance_id+"_M1.wav")
    flags,frame_counts = [], []
    for microphone,audio_path in [("m1",throat_path),("m2",path)]:
        info = sf.info(audio_path)
        assert info.samplerate == 16000 and info.channels == 1 and 0 < info.frames <= 480000
        frame_counts.append(info.frames)
        waveform,_ = sf.read(audio_path,dtype="float64")
        if np.sqrt(np.mean(waveform**2)) < .001: flags.append(microphone+"_rms_below_minus_60_dbfs")
        if np.mean((waveform <= -1) | (waveform >= 32767/32768)) > .001:
            flags.append(microphone+"_clipped_fraction_above_0_001")
    assert len(set(frame_counts)) == 1, "Paired microphone lengths differ"
    speaker_role = next(role for role,data in PARTITIONS.items() if speaker in data["speakers"])
    prompt_role = next(role for role,data in PARTITIONS.items() if prompt in data["prompts"])
    split = speaker_role if speaker_role == prompt_role else f"{speaker_role}_speaker_{prompt_role}_prompt"
    candidate_records.append({"utterance_id":utterance_id,"speaker":speaker,"session":session,
        "prompt":prompt,"split":split,"speaker_partition":speaker_role,"prompt_partition":prompt_role,
        "m1_path":str(throat_path.relative_to(corpus)),"m2_path":str(path.relative_to(corpus)),
        "m1_sha256":sha256(throat_path),"m2_sha256":sha256(path),
        "label_path":str(label.relative_to(corpus)),"label_sha256":sha256(label),
        "duration_seconds":frame_counts[0]/16000,"transcript_roman":roman,
        "draft_hindi":" ".join(hindi_words),"reference_hindi":"","review_status":"pending",
        "reviewed_by":"","reviewed_at":"","review_notes":"","signal_flags":flags})
candidate_records.sort(key=lambda row:row["utterance_id"])
assert len({r["utterance_id"] for r in candidate_records}) == len(candidate_records)
candidate_manifest = "".join(json.dumps(row,ensure_ascii=False,sort_keys=True)+"\n" for row in candidate_records)
if MANIFEST.exists():
    assert MANIFEST.read_text() == candidate_manifest, "Raw data/conversion changed; create a separate version"
    print("Existing manifest reproduced exactly; no labels or splits changed.")
else:
    MANIFEST.parent.mkdir(parents=True,exist_ok=True)
    MANIFEST.write_text(candidate_manifest,encoding="utf-8")
    summary = {"records":len(candidate_records),"speakers":len({r["speaker"] for r in candidate_records}),
        "minutes_per_channel":sum(r["duration_seconds"] for r in candidate_records)/60,
        "counts_by_split":dict(Counter(r["split"] for r in candidate_records)),"reviewed":0,
        "excluded":exclusions,"manifest_sha256":sha256(MANIFEST),"notebook_code_sha256":SOURCE_HASH,
        "reference_source":REFERENCE_SOURCE,"draft_corrections":["oora -> ओर","H10 ki -> की","H21 dara -> डर"]}
    write_json(MANIFEST.parent/"summary.json",summary)
    write_csv(MANIFEST.parent/"exclusions.csv",exclusions,["utterance_id","path","reason"])
print("Included:",len(candidate_records),"recordings; exclusions:",exclusions)


## Read the frozen manifest and verify separation

The manifest is the recording index. We use the accepted annotation version while retaining its original review fields. Training teaches the model; development chooses settings. Speaker and prompt sets must be disjoint for the primary train/dev/test groups.


In [ ]:
records = read_jsonl(MANIFEST)
assert len({row["utterance_id"] for row in records}) == len(records), "Duplicate recording IDs"
for row in records:
    row["text"] = (row["reference_hindi"] if row["review_status"] == "reviewed"
                   and row["reference_hindi"].strip() else row["draft_hindi"])
    assert row["text"].strip(), row["utterance_id"]
train = [row for row in records if row["split"] == "train"]
dev = [row for row in records if row["split"] == "dev"]
assert train and dev
for first, second in [("train", "dev"), ("train", "test"), ("dev", "test")]:
    for field in ["speaker", "prompt"]:
        a = {row[field] for row in records if row["split"] == first}
        b = {row[field] for row in records if row["split"] == second}
        assert a.isdisjoint(b), f"{field} overlap between {first} and {second}"
MANIFEST_HASH = sha256(MANIFEST)
for split in ["train", "dev", "test"]:
    selected = [row for row in records if row["split"] == split]
    print(split, len(selected), "recordings;", len({r["speaker"] for r in selected}), "speakers;",
          len({r["prompt"] for r in selected}), "prompts;",
          round(sum(r["duration_seconds"] for r in selected)/60, 3), "minutes")
print("Manifest fingerprint:", MANIFEST_HASH)


## Declare the development comparison

The three situations separate unfamiliar speakers, unfamiliar prompts, and both. An evaluation waveform must not appear in training. The new notebook declaration is stored separately from the old helper-based declaration. Neither is an untouched final-test protocol.


In [ ]:
PAPER.mkdir(parents=True, exist_ok=True)
training_audio = {row["m2_sha256"] for row in train}
group_counts = {}
for label, split in GROUPS.items():
    selected = [row for row in records if row["split"] == split]
    assert selected
    speaker_role, prompt_role = {"dev_speaker_train_prompt": ("dev", "train"),
        "train_speaker_dev_prompt": ("train", "dev"), "dev": ("dev", "dev")}[split]
    assert all(row["speaker_partition"] == speaker_role and row["prompt_partition"] == prompt_role
               for row in selected), "Evaluation partition mismatch"
    assert training_audio.isdisjoint({row["m2_sha256"] for row in selected}), "Training waveform in evaluation"
    group_counts[label] = {"split": split, "utterances": len(selected),
                          "speakers": len({r["speaker"] for r in selected}),
                          "prompts": len({r["prompt"] for r in selected})}
EXPECTED_SETTINGS = {"manifest_sha256": MANIFEST_HASH, "model": MODEL, "model_revision": REVISION,
                     "generation": GENERATION, "normalization": NORMALIZATION,
                     "microphone": "m2", "augmentation": "none", "allow_drafts": True}
protocol = {**EXPECTED_SETTINGS, "reference_source": REFERENCE_SOURCE, "groups": group_counts,
            "annotation_acceptance": "User accepted supplied labels; no new listening review claimed",
            "checkpoint_rule": "minimum primary development WER; ties keep earlier epoch",
            "learning_rate_tie_rule": "equal primary development WER chooses smaller rate",
            "scope": "development diagnostics; legacy test remains exploratory",
            "implementation": "explicit_notebooks_v1"}
protocol_path = PAPER/"notebook_protocol.json"
if protocol_path.exists():
    old = json.loads(protocol_path.read_text())
    assert {k:v for k,v in old.items() if k != "declared_at_utc"} == protocol, "Protocol changed; use a new version."
    protocol = old
else:
    protocol["declared_at_utc"] = datetime.now(timezone.utc).isoformat()
    write_json(protocol_path, protocol)
print("Declared development groups:", group_counts)


## Listen to one example

This is an illustration of the input and reference, not a new mandatory reference-review stage.


In [ ]:
example = dev[0]
print(example["utterance_id"],example["transcript_roman"],"->",example["text"])
display(Audio(filename=str(corpus_path(CORPUS,example["m2_path"]))))


## Calculate word and character errors

Normalization applies NFC Unicode, turns punctuation into spaces, and collapses whitespace, equally for references and predictions. The edit-distance loop counts substitutions, deletions, and insertions. Corpus WER sums errors over all reference words; it is not an average of recording percentages. CER here counts Unicode code points, including spaces, rather than visible Hindi graphemes.


In [ ]:
def normalize(text):
    text = unicodedata.normalize("NFC", text)
    text = "".join(" " if unicodedata.category(c).startswith("P") else c for c in text)
    return " ".join(text.split())

def edits(reference, hypothesis):
    # Levenshtein DP with deterministic tie priority: substitution, deletion, insertion.
    prev = [(j, 0, 0, j) for j in range(len(hypothesis) + 1)]
    for i, ref in enumerate(reference, 1):
        current = [(i, 0, i, 0)]
        for j, hyp in enumerate(hypothesis, 1):
            if ref == hyp:
                current.append(prev[j - 1])
                continue
            cost, s, d, ins = prev[j - 1]
            sub = (cost + 1, s + 1, d, ins)
            cost, s, d, ins = prev[j]
            delete = (cost + 1, s, d + 1, ins)
            cost, s, d, ins = current[j - 1]
            insert = (cost + 1, s, d, ins + 1)
            current.append(min((sub, delete, insert), key=lambda x: x[0]))
        prev = current
    _, s, d, ins = prev[-1]
    return {"substitutions": s, "deletions": d, "insertions": ins,
            "reference_units": len(reference), "errors": s + d + ins}

def summarize(rows, normalized=True):
    if not rows:
        raise ValueError("No predictions to score")
    counts = {"word": defaultdict(int), "character": defaultdict(int)}
    details = []
    for row in rows:
        ref, hyp = row["reference"], row["prediction"]
        if normalized:
            ref, hyp = normalize(ref), normalize(hyp)
        if not ref.strip():
            raise ValueError(f"Empty reference: {row.get('utterance_id', row.get('utterance'))}")
        detail = {"utterance_id": row.get("utterance_id", row.get("utterance")),
                  "speaker": row["speaker"], "prompt": row.get("prompt", "")}
        for unit, r, h in (("word", ref.split(), hyp.split()), ("character", list(ref), list(hyp))):
            values = edits(r, h)
            detail[unit] = values
            for k, v in values.items():
                counts[unit][k] += v
        details.append(detail)
    result = {"utterances": len(rows)}
    for unit, name in (("word", "wer_percent"), ("character", "cer_percent")):
        result[unit] = dict(counts[unit])
        result[name] = 100 * counts[unit]["errors"] / counts[unit]["reference_units"]
    return result, details

def score(rows):
    ids = [r.get("utterance_id", r.get("utterance")) for r in rows]
    if None in ids or len(set(ids)) != len(ids):
        raise ValueError("Prediction IDs must be present and unique")
    raw, _ = summarize(rows, normalized=False)
    normalized, details = summarize(rows)
    by_speaker, by_prompt = {}, {}
    for key, dest in (("speaker", by_speaker), ("prompt", by_prompt)):
        groups = defaultdict(list)
        for row in rows:
            groups[row.get(key, "")].append(row)
        for value, group in sorted(groups.items()):
            if value:
                dest[value] = summarize(group)[0]
    return {"normalization": NORMALIZATION,
            "cer_units": "Unicode code points, including spaces; not grapheme clusters",
            "raw": raw, "normalized": normalized, "by_speaker": by_speaker,
            "by_prompt": by_prompt,
            "macro_speaker_wer_percent": sum(r["wer_percent"] for r in by_speaker.values()) / len(by_speaker),
            "utterance_errors": details,
            "edit_alignment": "minimum distance; tie priority substitution, deletion, insertion"}


## Check the scorer on a simple example

One incorrect word among five reference words gives 20% WER. This small example explains the arithmetic, not model quality.


In [ ]:
example_score = score([{"utterance_id":"example","speaker":"example","prompt":"example",
    "reference":"मैं घर जा रहा हूं","prediction":"मैं घर आ रहा हूं"}])
assert example_score["normalized"]["wer_percent"] == 20.0
print(example_score["normalized"]["word"],"WER:",example_score["normalized"]["wer_percent"])


## Load Whisper, generate Hindi text, and score the baseline

You can see audio loading, feature extraction, tensor stacking, model.generate, decoding, and CSV scoring below. If the completed baseline matches the data/model/settings, its predictions are verified and reused. No optimizer or weight update occurs.


In [ ]:
run_name = "baseline_dev"
directory = ROOT/"runs"/run_name
configuration = {"split": "dev", "batch_size": 8}
if (directory/"metadata.json").exists():
    metadata = json.loads((directory/"metadata.json").read_text())
    for key, value in {**EXPECTED_SETTINGS, "configuration": configuration}.items():
        assert metadata.get(key) == value, f"Baseline changed: {key}"
    assert (metadata.get("utils_sha256") == LEGACY_HELPER_SHA256
            or metadata.get("notebook_code_sha256") in COMPATIBLE_CODE_HASHES[NOTEBOOK_NAME]), "Unknown baseline code"
    assert sha256(directory/"predictions.csv") == metadata["predictions_sha256"]
    print("Reusing saved baseline.")
else:
    started = time.monotonic()
    torch.cuda.reset_peak_memory_stats()
    try:
        base_path = snapshot_download(MODEL, revision=REVISION, local_files_only=True)
        base_revision = None  # The local directory already identifies the pinned revision.
    except LocalEntryNotFoundError:
        base_path, base_revision = MODEL, REVISION

    processor = WhisperProcessor.from_pretrained(base_path, revision=base_revision, language="hindi", task="transcribe")
    processor.tokenizer.set_prefix_tokens(language="hindi", task="transcribe", predict_timestamps=False)
    model = WhisperForConditionalGeneration.from_pretrained(base_path, revision=base_revision, use_safetensors=True)
    model.generation_config.forced_decoder_ids = None
    model.generation_config.language = "hi"
    model.generation_config.task = "transcribe"
    model.to("cuda")
    evaluation_records = dev
    model.eval()
    predictions = []
    for start in range(0, len(evaluation_records), 8):
        batch_records = evaluation_records[start:start + 8]
        features, masks = [], []
        for record in batch_records:
            audio, sample_rate = sf.read(corpus_path(CORPUS, record["m2_path"]), dtype="float32")
            assert sample_rate == 16000 and audio.ndim == 1 and 0 < len(audio) <= 480000
            encoded = processor.feature_extractor(audio, sampling_rate=sample_rate, return_attention_mask=True)
            features.append(torch.as_tensor(encoded["input_features"][0]))
            masks.append(torch.as_tensor(encoded["attention_mask"][0]))
        features = torch.stack(features).to("cuda")
        masks = torch.stack(masks).to("cuda")
        with torch.inference_mode():
            token_ids = model.generate(input_features=features, attention_mask=masks, **GENERATION)
        texts = processor.batch_decode(token_ids, skip_special_tokens=True)
        for record, text, tokens in zip(batch_records, texts, token_ids):
            predictions.append({"utterance_id": record["utterance_id"], "speaker": record["speaker"],
                "prompt": record["prompt"], "reference": record["text"], "prediction": text,
                "reference_status": record["review_status"], "audio_sha256": record["m2_sha256"],
                "generation_cap_reached": bool(len(tokens) >= GENERATION["max_new_tokens"] + 1
                    and processor.tokenizer.eos_token_id not in tokens[1:].tolist())})
    directory.mkdir(parents=True, exist_ok=True)
    write_csv(directory/"predictions.csv", predictions, PREDICTION_FIELDS)
    report = score(read_csv(directory/"predictions.csv"))
    report["generation_cap_count"] = sum(str(r["generation_cap_reached"]).lower() == "true" for r in predictions)
    report["predictions_sha256"] = sha256(directory/"predictions.csv")
    write_json(directory/"metrics.json", report)

    metadata = {**EXPECTED_SETTINGS, "configuration": configuration, "environment": ENVIRONMENT,
                "implementation": "explicit_notebooks_v1", "notebook": NOTEBOOK_NAME,
                "notebook_code_sha256": SOURCE_HASH, "notebook_hash_format": "python_ast_v1", "protocol": protocol,
                "elapsed_seconds": time.monotonic()-started,
                "peak_gpu_allocated_bytes": torch.cuda.max_memory_allocated(),
                "predictions_sha256": report["predictions_sha256"]}
    write_json(directory/"metadata.json", metadata)
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
predictions = read_csv(directory/"predictions.csv")
assert {r["utterance_id"] for r in predictions} == {r["utterance_id"] for r in dev}
assert len(predictions) == len(dev)
by_id = {r["utterance_id"]:r for r in dev}
for row in predictions:
    record = by_id[row["utterance_id"]]
    assert row["reference"] == record["text"] and row["audio_sha256"] == record["m2_sha256"]
report = score(predictions)
assert report["normalized"] == json.loads((directory/"metrics.json").read_text())["normalized"]
print(json.dumps(report["normalized"], ensure_ascii=False, indent=2))
RESEARCH_RUNS = [run_name]


## Read individual recognition errors

The reference is the expected text; the prediction is Whisper's output. Inspect actual substitutions, missing words, and extra words rather than relying only on the aggregate number.


In [ ]:
for row in read_csv(ROOT/"runs/baseline_dev/predictions.csv")[:5]:
    print(row["utterance_id"],"\nREF:",row["reference"],"\nPRED:",row["prediction"],"\n")


## Evaluate the reserved development groups

The complete model-loading and transcription loops are below. Old compatible predictions are checked and reused; a missing group triggers real GPU inference. These case differences are descriptive because speakers/content differ.


In [ ]:
# Only these development cases run. No legacy test is generated here.
for run_name in RESEARCH_RUNS:
    run_folder = ROOT/"runs"/run_name
    metadata = json.loads((run_folder/"metadata.json").read_text())
    for key,value in EXPECTED_SETTINGS.items():
        assert metadata.get(key) == value, f"Run identity changed: {key}"
    assert (metadata.get("utils_sha256") == LEGACY_HELPER_SHA256
            or metadata.get("notebook_code_sha256") in COMPATIBLE_CODE_HASHES.get(metadata.get("notebook"), set())), "Unknown run code"
    assert "execution_check_train_ids" not in metadata["configuration"], "Tiny check is not a research result"
    if "mode" in metadata["configuration"]:
        assert metadata["status"] == "completed"
    assert metadata["predictions_sha256"] == sha256(run_folder/"predictions.csv")
    if "mode" in metadata["configuration"]:
        checkpoint = run_folder/"best"
        files = sorted(checkpoint.glob("*.safetensors")) + sorted(checkpoint.glob("*config.json"))
        assert any(path.suffix == ".safetensors" for path in files), "Missing selected weights"
        checkpoint_identity = {path.name:sha256(path) for path in files}
    else:
        checkpoint_identity = {"pretrained_revision":REVISION}
    model, processor = None, None
    try:
        for label,split in GROUPS.items():
            evaluation_records = [row for row in records if row["split"] == split]
            folder = run_folder if split == "dev" else run_folder/("evaluation_"+split)
            cached = (folder/"metadata.json").exists()
            if cached:
                if split != "dev":
                    saved = json.loads((folder/"metadata.json").read_text())
                    signature = saved["signature"]
                    assert signature["checkpoint"] == checkpoint_identity, "Selected weights changed"
                    assert signature["configuration"] == metadata["configuration"]
                    assert signature["selected_predictions_sha256"] == metadata["predictions_sha256"]
                    assert signature["split"] == split and signature["batch_size"] == 8
                    saved_protocol = signature["protocol"]
                    for key in ["manifest_sha256","model_revision","generation","normalization","groups"]:
                        assert saved_protocol[key] == protocol[key], f"Cached protocol differs: {key}"
                    if "research_sha256" in saved_protocol:
                        assert saved_protocol["research_sha256"] == LEGACY_RESEARCH_SHA256
                        assert saved_protocol["utils_sha256"] == LEGACY_HELPER_SHA256
                    else:
                        assert saved_protocol == protocol
                    assert saved["predictions_sha256"] == sha256(folder/"predictions.csv")
                predictions = read_csv(folder/"predictions.csv")
            else:
                if model is None:
                    try:
                        base_path = snapshot_download(MODEL, revision=REVISION, local_files_only=True)
                        base_revision = None  # The local directory already identifies the pinned revision.
                    except LocalEntryNotFoundError:
                        base_path, base_revision = MODEL, REVISION

                    if metadata["configuration"].get("mode") == "full":
                        model_path,model_revision = str(run_folder/"best"),None
                    else:
                        model_path,model_revision = base_path,base_revision
                    processor = WhisperProcessor.from_pretrained(model_path,revision=model_revision,language="hindi",task="transcribe")
                    processor.tokenizer.set_prefix_tokens(language="hindi",task="transcribe",predict_timestamps=False)
                    model = WhisperForConditionalGeneration.from_pretrained(model_path,revision=model_revision,use_safetensors=True)
                    if metadata["configuration"].get("mode") == "lora":
                        from peft import PeftModel
                        model = PeftModel.from_pretrained(model,run_folder/"best",is_trainable=False)
                    model.generation_config.forced_decoder_ids = None
                    model.generation_config.language = "hi"
                    model.generation_config.task = "transcribe"
                    model.to("cuda")
                print("Transcribing:",run_name,split,len(evaluation_records),"recordings",flush=True)
                inference_started = time.monotonic()
                model.eval()
                predictions = []
                for start in range(0, len(evaluation_records), 8):
                    batch_records = evaluation_records[start:start + 8]
                    features, masks = [], []
                    for record in batch_records:
                        audio, sample_rate = sf.read(corpus_path(CORPUS, record["m2_path"]), dtype="float32")
                        assert sample_rate == 16000 and audio.ndim == 1 and 0 < len(audio) <= 480000
                        encoded = processor.feature_extractor(audio, sampling_rate=sample_rate, return_attention_mask=True)
                        features.append(torch.as_tensor(encoded["input_features"][0]))
                        masks.append(torch.as_tensor(encoded["attention_mask"][0]))
                    features = torch.stack(features).to("cuda")
                    masks = torch.stack(masks).to("cuda")
                    with torch.inference_mode():
                        token_ids = model.generate(input_features=features, attention_mask=masks, **GENERATION)
                    texts = processor.batch_decode(token_ids, skip_special_tokens=True)
                    for record, text, tokens in zip(batch_records, texts, token_ids):
                        predictions.append({"utterance_id": record["utterance_id"], "speaker": record["speaker"],
                            "prompt": record["prompt"], "reference": record["text"], "prediction": text,
                            "reference_status": record["review_status"], "audio_sha256": record["m2_sha256"],
                            "generation_cap_reached": bool(len(tokens) >= GENERATION["max_new_tokens"] + 1
                                and processor.tokenizer.eos_token_id not in tokens[1:].tolist())})
                folder.mkdir(parents=True, exist_ok=True)
                write_csv(folder/"predictions.csv", predictions, PREDICTION_FIELDS)
                report = score(read_csv(folder/"predictions.csv"))
                report["generation_cap_count"] = sum(str(r["generation_cap_reached"]).lower() == "true" for r in predictions)
                report["predictions_sha256"] = sha256(folder/"predictions.csv")
                write_json(folder/"metrics.json", report)

                signature = {"protocol":protocol,"checkpoint":checkpoint_identity,
                             "selected_predictions_sha256":metadata["predictions_sha256"],
                             "configuration":metadata["configuration"],"split":split,"batch_size":8}
                write_json(folder/"metadata.json",{"signature":signature,
                    "predictions_sha256":sha256(folder/"predictions.csv"),"notebook_code_sha256":SOURCE_HASH,
                    "elapsed_seconds":time.monotonic()-inference_started,"environment":ENVIRONMENT})
            expected = {row["utterance_id"]:row for row in evaluation_records}
            assert len(predictions) == len(expected) and {r["utterance_id"] for r in predictions} == set(expected)
            for row in predictions:
                original = expected[row["utterance_id"]]
                assert row["reference"] == original["text"] and row["audio_sha256"] == original["m2_sha256"]
                assert row["speaker"] == original["speaker"] and row["prompt"] == original["prompt"]
            report = score(predictions)
            assert report["normalized"] == json.loads((folder/"metrics.json").read_text())["normalized"]
            print(run_name,label,"WER:",round(report["normalized"]["wer_percent"],2))
    finally:
        if model is not None:
            del model,processor
            gc.collect()
            torch.cuda.empty_cache()


## Save the paper tables

All scores come from the exact exported predictions. Different configurations remain separate; one seed does not have a measured seed standard deviation. We preserve the old provenance file and add notebook_table_sources.json.


In [ ]:
# Recalculate tables from exact saved CSVs, keeping every completed configuration.
run_rows, group_rows, speaker_rows, seed_groups, sources = [], [], [], {}, {}
for directory in sorted((ROOT/"runs").glob("*")):
    if not (directory/"metadata.json").is_file() or not (directory/"metrics.json").is_file():
        continue
    metadata = json.loads((directory/"metadata.json").read_text())
    config = metadata["configuration"]
    if "execution_check_train_ids" in config or ("mode" in config and metadata.get("status") != "completed"):
        continue
    for key,value in EXPECTED_SETTINGS.items():
        assert metadata.get(key) == value, f"Incompatible table source: {directory.name}/{key}"
    assert (metadata.get("utils_sha256") == LEGACY_HELPER_SHA256
            or metadata.get("notebook_code_sha256") in COMPATIBLE_CODE_HASHES.get(metadata.get("notebook"), set())), "Unknown table source code"
    assert sha256(directory/"predictions.csv") == metadata["predictions_sha256"]
    primary = score(read_csv(directory/"predictions.csv"))
    assert primary["normalized"] == json.loads((directory/"metrics.json").read_text())["normalized"]
    run_rows.append({"run":directory.name,"mode":config.get("mode","pretrained"),
        "learning_rate":config.get("learning_rate",""),"seed":config.get("seed",""),
        "best_epoch":metadata.get("best_epoch",""),"wer_percent":primary["normalized"]["wer_percent"],
        "cer_percent":primary["normalized"]["cer_percent"],"trainable_parameters":metadata.get("trainable_parameters",0),
        "elapsed_minutes":metadata["elapsed_seconds"]/60,"peak_gpu_allocated_gib":metadata["peak_gpu_allocated_bytes"]/2**30})
    sources[directory.name] = {"metadata_sha256":sha256(directory/"metadata.json"),
        "primary_predictions_sha256":sha256(directory/"predictions.csv"),"evaluation_predictions_sha256":{}}
    weights_identity = None
    for label,split in GROUPS.items():
        folder = directory if split == "dev" else directory/("evaluation_"+split)
        if not (folder/"predictions.csv").exists():
            continue
        rows = read_csv(folder/"predictions.csv")
        expected = {r["utterance_id"]:r for r in records if r["split"] == split}
        assert len(rows) == len(expected) and {r["utterance_id"] for r in rows} == set(expected)
        for row in rows:
            reference = expected[row["utterance_id"]]
            assert row["reference"] == reference["text"] and row["audio_sha256"] == reference["m2_sha256"]
            assert row["speaker"] == reference["speaker"] and row["prompt"] == reference["prompt"]
        if split != "dev":
            cached = json.loads((folder/"metadata.json").read_text())
            assert cached["predictions_sha256"] == sha256(folder/"predictions.csv")
            signature = cached["signature"]
            assert signature["selected_predictions_sha256"] == metadata["predictions_sha256"]
            assert signature["configuration"] == config and signature["split"] == split and signature["batch_size"] == 8
            for key in ["manifest_sha256","model_revision","generation","normalization","groups"]:
                assert signature["protocol"][key] == protocol[key]
            if "research_sha256" in signature["protocol"]:
                assert signature["protocol"]["research_sha256"] == LEGACY_RESEARCH_SHA256
                assert signature["protocol"]["utils_sha256"] == LEGACY_HELPER_SHA256
            else:
                assert signature["protocol"] == protocol
            if weights_identity is None:
                if "mode" in config:
                    checkpoint_files = sorted((directory/"best").glob("*.safetensors")) + sorted((directory/"best").glob("*config.json"))
                    assert any(p.suffix == ".safetensors" for p in checkpoint_files)
                    weights_identity = {p.name:sha256(p) for p in checkpoint_files}
                else:
                    weights_identity = {"pretrained_revision":REVISION}
            assert signature["checkpoint"] == weights_identity
            sources[directory.name]["evaluation_predictions_sha256"][split] = sha256(folder/"predictions.csv")
        report = score(rows)
        assert report["normalized"] == json.loads((folder/"metrics.json").read_text())["normalized"]
        group_rows.append({"run":directory.name,"group":label,"utterances":len(rows),
            "wer_percent":report["normalized"]["wer_percent"],"cer_percent":report["normalized"]["cer_percent"],
            **report["normalized"]["word"]})
        for speaker,values in report["by_speaker"].items():
            speaker_rows.append({"run":directory.name,"group":label,"speaker":speaker,
                "utterances":values["utterances"],"wer_percent":values["wer_percent"],
                "cer_percent":values["cer_percent"],**values["word"]})
    if "mode" in config:
        key = json.dumps({k:v for k,v in config.items() if k != "seed"},sort_keys=True)
        per_seed = seed_groups.setdefault(key,{})
        assert config["seed"] not in per_seed, "Duplicate configuration/seed is not independent evidence"
        per_seed[config["seed"]] = primary["normalized"]
seed_rows = []
for key,per_seed in seed_groups.items():
    config = json.loads(key)
    row = {"mode":config["mode"],"learning_rate":config["learning_rate"],"configuration":key,
           "seed_count":len(per_seed),"seeds":" ".join(map(str,sorted(per_seed)))}
    for metric in ["wer_percent","cer_percent"]:
        values = [r[metric] for r in per_seed.values()]
        row["mean_"+metric] = statistics.mean(values)
        row["std_"+metric] = statistics.stdev(values) if len(values) > 1 else ""
    seed_rows.append(row)
for name,rows in [("development_runs",run_rows),("generalization",group_rows),("speakers",speaker_rows),("seed_summary",seed_rows)]:
    if rows:
        write_csv(PAPER/(name+".csv"),rows,list(rows[0]))
# Preserve the earlier helper-based provenance file and write the new implementation separately.
write_json(PAPER/"notebook_table_sources.json",{"manifest_sha256":MANIFEST_HASH,
    "notebook_code_sha256":SOURCE_HASH,"notebook_code_hashes":NOTEBOOK_CODE_HASHES,
    "reference_source":REFERENCE_SOURCE,"runs":sources})
for row in group_rows:
    print(row["run"],row["group"],"WER:",round(row["wer_percent"],2))
print("Seed summaries:",json.dumps(seed_rows,indent=2))
print("Saved CSV tables:",PAPER)


## Next research step

Both learning-rate searches, three selected seeds per method, and final_v3 exploratory holdout inference are complete and checked. Read the final comparison below and EVALUATION_PROTOCOL.md. Use all seeds and all three cases; the previously inspected legacy test is not confirmatory evidence. Next examine speaker/prompt sensitivity, prepare the paper tables/figures and methods/results text, and predefine additional partitions for stronger evidence if feasible. No augmentation is included.


## Final evaluation — frozen exploratory holdouts

The training above is already complete. This section only loads selected checkpoints and transcribes held-out recordings. Read `EVALUATION_PROTOCOL.md` first. We compare 189 unfamiliar-speaker/familiar-sentence recordings, 213 familiar-speaker/unfamiliar-sentence recordings, and 45 both-unfamiliar recordings. The original test was inspected earlier, so these are exploratory results, not an untouched confirmatory test.

For evaluation only after a restart, run the earlier **Locate the project**, **Read/write files**, **Code identity**, **Read the frozen manifest**, and **Calculate word and character errors** cells, then this section. No training cell is needed. Outputs have a separate folder, `runs/final_v3/`; old training and development results remain intact.


### Enable evaluation deliberately

The default is disabled. Set the following flag to `True` when you intend to run inference. This flag does not start training or select a new model. Notebook 01 evaluates the baseline; notebook 02 evaluates all three full checkpoints; notebook 03 evaluates all three LoRA checkpoints.

The saved evaluation outputs below were generated with the flag enabled in an isolated execution kernel. The saved flag remains `False` to avoid accidental inference.


In [ ]:
"""HDSD final evaluation v3: controls."""
RUN_FINAL_EVALUATION = False  # Explicit opt-in. Does not retrain any model.


### Check exactly what will be evaluated

Before reading results, check the frozen protocol, data, software environment, checkpoint files, and the evaluation code. A hash identifies exact file contents. These checks stop accidental mixing of incompatible experiments. Evaluation code is recorded separately so adding this section does not relabel completed training.


In [ ]:
"""HDSD final evaluation v3: verify the frozen protocol and selected checkpoints."""
if RUN_FINAL_EVALUATION:
    FINAL_ROOT = ROOT/"runs/final_v3"
    final_protocol_path = FINAL_ROOT/"protocol.json"
    FINAL_PROTOCOL_HASH = sha256(final_protocol_path)
    assert FINAL_PROTOCOL_HASH == (FINAL_ROOT/"protocol.sha256").read_text().split()[0], "Protocol file changed"
    FINAL_PROTOCOL = json.loads(final_protocol_path.read_text())
    assert FINAL_PROTOCOL["version"] == "final_v3"
    assert sha256(ROOT/"EVALUATION_PROTOCOL.md") == FINAL_PROTOCOL["human_protocol_sha256"]
    assert MANIFEST_HASH == FINAL_PROTOCOL["manifest_sha256"] == sha256(MANIFEST)
    assert MODEL == FINAL_PROTOCOL["model"] and REVISION == FINAL_PROTOCOL["model_revision"]
    assert GENERATION == FINAL_PROTOCOL["generation"] and NORMALIZATION == FINAL_PROTOCOL["normalization"]
    assert NOTEBOOK_CODE_HASHES == FINAL_PROTOCOL["scientific_engine_sha256"], "Preparation/training/scoring code changed"
    assert ENVIRONMENT == FINAL_PROTOCOL["inference_environment"], "Use the frozen cluster environment"

    # Evaluation code has its own identity, separate from already completed training.
    for filename in NOTEBOOK_FILES:
        notebook = json.loads((ROOT/filename).read_text())
        statements = []
        for cell in notebook["cells"]:
            if cell["cell_type"] != "code":
                continue
            tree = ast.parse("".join(cell["source"]))
            if not (tree.body and isinstance(tree.body[0], ast.Expr)
                    and isinstance(tree.body[0].value, ast.Constant)
                    and isinstance(tree.body[0].value.value, str)
                    and tree.body[0].value.value.startswith("HDSD final evaluation v3:")):
                continue
            if tree.body[0].value.value == "HDSD final evaluation v3: controls.":
                assert len(tree.body) == 2 and isinstance(tree.body[1], ast.Assign)
                assert [target.id for target in tree.body[1].targets] == ["RUN_FINAL_EVALUATION"]
                assert isinstance(tree.body[1].value, ast.Constant) and type(tree.body[1].value.value) is bool
                continue
            statements.extend(syntax_signature(node) for node in tree.body)
        digest = hashlib.sha256(json.dumps(statements,ensure_ascii=False,sort_keys=True).encode()).hexdigest()
        assert digest == FINAL_PROTOCOL["evaluation_code_sha256"][filename], "Evaluation code changed"

    original_by_id = {row["utterance_id"]:row for row in records}
    training_speakers = {row["speaker"] for row in train}
    training_prompts = {row["prompt"] for row in train}
    training_audio = {row[key] for row in train for key in ["m1_sha256","m2_sha256"]}
    final_ids = set()
    for split, group in FINAL_PROTOCOL["groups"].items():
        selected = [row for row in records if row["split"] == split]
        assert [row["utterance_id"] for row in selected] == [row["utterance_id"] for row in group["records"]]
        assert len(selected) == group["utterances"] and final_ids.isdisjoint(row["utterance_id"] for row in selected)
        final_ids.update(row["utterance_id"] for row in selected)
        for original, frozen in zip(selected,group["records"]):
            assert all(original[key] == value for key,value in frozen.items()), original["utterance_id"]
            assert original["speaker_partition"] == group["speaker_partition"]
            assert original["prompt_partition"] == group["prompt_partition"]
            assert original["m1_sha256"] not in training_audio and original["m2_sha256"] not in training_audio
        speakers, prompts = {r["speaker"] for r in selected}, {r["prompt"] for r in selected}
        assert len(speakers) == group["speakers"] and len(prompts) == group["prompts"]
        assert speakers <= training_speakers if group["speaker_partition"] == "train" else speakers.isdisjoint(training_speakers)
        assert prompts <= training_prompts if group["prompt_partition"] == "train" else prompts.isdisjoint(training_prompts)

    evaluation_mode = {NOTEBOOK_FILES[0]:"pretrained", NOTEBOOK_FILES[1]:"full", NOTEBOOK_FILES[2]:"lora"}[NOTEBOOK_NAME]
    FINAL_MODELS = [item for item in FINAL_PROTOCOL["models"] if item["mode"] == evaluation_mode]
    for item in FINAL_MODELS:
        folder = ROOT/"runs"/item["run"]
        assert sha256(folder/"metadata.json") == item["metadata_sha256"], "Training record changed"
        metadata = json.loads((folder/"metadata.json").read_text())
        assert metadata["configuration"] == item["configuration"]
        assert sha256(folder/"predictions.csv") == item["development_predictions_sha256"]
        if item["mode"] != "pretrained":
            assert metadata["status"] == "completed" and metadata["finished_epochs"] == 5
            assert metadata["best_epoch"] == item["best_epoch"]
            assert {p.name:sha256(p) for p in (folder/"best").iterdir() if p.is_file()} == item["checkpoint_files"]
    print("Frozen protocol:", FINAL_PROTOCOL_HASH)
    print("Evaluation status:", FINAL_PROTOCOL["status"])
    print("Models:",[item["run"] for item in FINAL_MODELS])
    print("Groups:",{key:value["utterances"] for key,value in FINAL_PROTOCOL["groups"].items()})
else:
    print("Final evaluation is disabled. Completed training and development results are preserved.")


### Load each saved model and transcribe the recordings

The loop below shows every operation: load the pinned model or adapter, read and verify M2 audio, extract features, generate Hindi tokens, decode text, and save CSV predictions. `torch.inference_mode()` disables gradient calculation; no optimizer is created and no weights are updated. Greedy decoding uses the same settings for every model. Completed compatible groups are checked and reused after interruption. A group is committed only when all its files are ready.


In [ ]:
"""HDSD final evaluation v3: generate predictions from selected checkpoints without training."""
if RUN_FINAL_EVALUATION:
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = True
    for item in FINAL_MODELS:
        model, processor = None, None
        loading_seconds = 0.
        try:
            for split, group in FINAL_PROTOCOL["groups"].items():
                folder = FINAL_ROOT/item["run"]/split
                signature = {"protocol_sha256":FINAL_PROTOCOL_HASH,"model":item,
                             "split":split,"batch_size":8,"dtype":"float32"}
                if folder.exists():
                    assert (folder/"metadata.json").is_file(), "Incomplete result directory; inspect before continuing"
                    cached = json.loads((folder/"metadata.json").read_text())
                    assert cached["signature"] == signature, "Cached evaluation belongs to different inputs"
                    assert cached["predictions_sha256"] == sha256(folder/"predictions.csv")
                    print("Reusing:",item["run"],split,flush=True)
                else:
                    if model is None:
                        loading_started = time.monotonic()
                        base_path = snapshot_download(MODEL, revision=REVISION, local_files_only=True)
                        model_path = str(ROOT/"runs"/item["run"]/"best") if item["mode"] == "full" else base_path
                        processor = WhisperProcessor.from_pretrained(model_path,language="hindi",task="transcribe",local_files_only=True)
                        processor.tokenizer.set_prefix_tokens(language="hindi",task="transcribe",predict_timestamps=False)
                        model = WhisperForConditionalGeneration.from_pretrained(model_path,use_safetensors=True,local_files_only=True)
                        if item["mode"] == "lora":
                            from peft import PeftModel
                            model = PeftModel.from_pretrained(model,ROOT/"runs"/item["run"]/"best",is_trainable=False,local_files_only=True)
                        prefix = [model.config.decoder_start_token_id] + processor.tokenizer.convert_tokens_to_ids(
                            ["<|hi|>","<|transcribe|>","<|notimestamps|>"])
                        assert processor.tokenizer.prefix_tokens == prefix, "Unexpected Hindi decoder prefix"
                        # Saved full checkpoints inherited different special-token defaults.
                        # Load the same pinned decoder configuration for every model.
                        from transformers import GenerationConfig
                        assert sha256(Path(base_path)/"generation_config.json") == FINAL_PROTOCOL["base_generation_config_sha256"]
                        model.generation_config = GenerationConfig.from_pretrained(base_path,local_files_only=True)
                        for key,value in GENERATION.items():
                            setattr(model.generation_config,key,value)
                        model.generation_config.forced_decoder_ids = None
                        model.generation_config.language = "hi"
                        model.generation_config.task = "transcribe"
                        resolved_generation = model.generation_config.to_dict()
                        assert resolved_generation == FINAL_PROTOCOL["resolved_generation_config"], "Decoder defaults differ"
                        resolved_generation_hash = hashlib.sha256(json.dumps(resolved_generation,ensure_ascii=False,sort_keys=True).encode()).hexdigest()
                        model.config.use_cache = True
                        model.to(device="cuda",dtype=torch.float32)
                        model.eval()
                        torch.cuda.synchronize()
                        loading_seconds = time.monotonic()-loading_started
                        print("Loaded:",item["run"],"Hindi prefix:",prefix,flush=True)
                    torch.cuda.reset_peak_memory_stats()
                    inference_started = time.monotonic()
                    predictions = []
                    for start in range(0,len(group["records"]),8):
                        batch_records = group["records"][start:start+8]
                        features, masks = [], []
                        for record in batch_records:
                            audio_path = corpus_path(CORPUS,record["m2_path"])
                            assert sha256(audio_path) == record["m2_sha256"], "Audio file changed"
                            audio, sample_rate = sf.read(audio_path,dtype="float32")
                            assert sample_rate == 16000 and audio.ndim == 1 and 0 < len(audio) <= 480000
                            encoded = processor.feature_extractor(audio,sampling_rate=sample_rate,return_attention_mask=True)
                            features.append(torch.as_tensor(encoded["input_features"][0]))
                            masks.append(torch.as_tensor(encoded["attention_mask"][0]))
                        with torch.inference_mode():
                            token_ids = model.generate(input_features=torch.stack(features).to("cuda"),
                                attention_mask=torch.stack(masks).to("cuda"),generation_config=model.generation_config,**GENERATION)
                        texts = processor.batch_decode(token_ids,skip_special_tokens=True)
                        assert len(texts) == len(batch_records)
                        for record,text,tokens in zip(batch_records,texts,token_ids):
                            predictions.append({"utterance_id":record["utterance_id"],"speaker":record["speaker"],
                                "prompt":record["prompt"],"reference":record["text"],"prediction":text,
                                "reference_status":record["review_status"],"audio_sha256":record["m2_sha256"],
                                "generation_cap_reached":bool(len(tokens) >= GENERATION["max_new_tokens"]+1
                                    and processor.tokenizer.eos_token_id not in tokens[1:].tolist())})
                        if (start//8+1)%10 == 0 or start+8 >= len(group["records"]):
                            print(item["run"],split,min(start+8,len(group["records"])),"/",len(group["records"]),flush=True)
                    torch.cuda.synchronize()
                    elapsed = time.monotonic()-inference_started
                    # Publish a group only after its CSV, metrics, and identity are all ready.
                    staging = folder.with_name(folder.name+".partial")
                    staging.mkdir(parents=True,exist_ok=True)
                    write_csv(staging/"predictions.csv",predictions,PREDICTION_FIELDS)
                    saved_rows = read_csv(staging/"predictions.csv")
                    report = score(saved_rows)
                    report["generation_cap_count"] = sum(row["generation_cap_reached"].lower()=="true" for row in saved_rows)
                    report["predictions_sha256"] = sha256(staging/"predictions.csv")
                    write_json(staging/"metrics.json",report)
                    write_json(staging/"metadata.json",{"signature":signature,"predictions_sha256":report["predictions_sha256"],
                        "created_at_utc":datetime.now(timezone.utc).isoformat(),"inference_seconds":elapsed,
                        "model_loading_seconds":loading_seconds,"inference_peak_allocated_bytes":torch.cuda.max_memory_allocated(),
                        "environment":ENVIRONMENT,"new_training":False,"new_inference":True,"resolved_generation_config_sha256":resolved_generation_hash})
                    staging.replace(folder)
                saved_rows = read_csv(folder/"predictions.csv")
                expected = {row["utterance_id"]:row for row in group["records"]}
                assert len(saved_rows) == len(expected) and {r["utterance_id"] for r in saved_rows} == set(expected)
                for row in saved_rows:
                    original = expected[row["utterance_id"]]
                    assert row["reference"] == original["text"] and row["audio_sha256"] == original["m2_sha256"]
                    assert row["speaker"] == original["speaker"] and row["prompt"] == original["prompt"]
                    assert row["reference_status"] == original["review_status"]
                recalculated = score(saved_rows)
                saved_report = json.loads((folder/"metrics.json").read_text())
                assert recalculated == {k:v for k,v in saved_report.items() if k not in ["predictions_sha256","generation_cap_count"]}
                assert saved_report["predictions_sha256"] == sha256(folder/"predictions.csv")
                assert saved_report["generation_cap_count"] == sum(row["generation_cap_reached"].lower()=="true" for row in saved_rows)
                print(item["run"],split,"WER:",round(recalculated["normalized"]["wer_percent"],2),flush=True)
        finally:
            if model is not None:
                del model,processor
                gc.collect()
                torch.cuda.empty_cache()


### Calculate comparisons from the saved CSVs

The complete table code is below. It waits until all seven models have finished. WER is total word edits divided by reference words, not the average of sentence WERs. Adapted-model summaries average three corpus scores; their standard deviation describes training-seed variation. The baseline is evaluated once.

For uncertainty, resample the same speakers for both methods 20,000 times and recompute the difference using all recordings belonging to each sampled speaker. Average the fixed trained seeds before this speaker resampling. The resulting percentile interval is conditional on these sentences and models, not a joint estimate of all training and population uncertainty. Also remove one prompt at a time to show sentence sensitivity. Positive left-minus-right differences favor the right model. Do not turn these exploratory intervals into a clinical or universal superiority claim.

Efficiency rows retain recorded end-to-end adaptation time, which includes preparation, development inference and checkpoint saving; it is not pure training throughput. LoRA checkpoint bytes describe an adapter that still needs the base model. Transcript-containing outputs stay private under the dataset agreement.


In [ ]:
"""HDSD final evaluation v3: compare exact saved predictions and paired speaker uncertainty."""
if RUN_FINAL_EVALUATION:
    pending = [(item["run"],split) for item in FINAL_PROTOCOL["models"] for split in FINAL_PROTOCOL["groups"]
               if not (FINAL_ROOT/item["run"]/split/"metadata.json").is_file()]
    if pending:
        print("Comparison waits for remaining notebook evaluations:",pending)
    else:
        reports, source_hashes, run_rows, speaker_rows, prompt_rows, error_rows = {}, {}, [], [], [], []
        for item in FINAL_PROTOCOL["models"]:
            for split,group in FINAL_PROTOCOL["groups"].items():
                folder = FINAL_ROOT/item["run"]/split
                metadata = json.loads((folder/"metadata.json").read_text())
                expected_signature = {"protocol_sha256":FINAL_PROTOCOL_HASH,"model":item,"split":split,"batch_size":8,"dtype":"float32"}
                assert metadata["signature"] == expected_signature
                assert metadata["resolved_generation_config_sha256"] == FINAL_PROTOCOL["resolved_generation_config_sha256"]
                assert sha256(folder/"predictions.csv") == metadata["predictions_sha256"]
                rows = read_csv(folder/"predictions.csv")
                expected = {r["utterance_id"]:r for r in group["records"]}
                assert len(rows) == len(expected) and {r["utterance_id"] for r in rows} == set(expected)
                for row in rows:
                    original = expected[row["utterance_id"]]
                    assert row["reference"] == original["text"] and row["audio_sha256"] == original["m2_sha256"]
                    assert row["speaker"] == original["speaker"] and row["prompt"] == original["prompt"]
                report = score(rows)
                saved = json.loads((folder/"metrics.json").read_text())
                assert report == {k:v for k,v in saved.items() if k not in ["predictions_sha256","generation_cap_count"]}
                assert saved["predictions_sha256"] == metadata["predictions_sha256"]
                assert saved["generation_cap_count"] == sum(r["generation_cap_reached"].lower()=="true" for r in rows)
                reports[(item["run"],split)] = report
                source_hashes[item["run"]+"/"+split] = {filename:sha256(folder/filename)
                    for filename in ["predictions.csv","metrics.json","metadata.json"]}
                values = report["normalized"]
                run_rows.append({"mode":item["mode"],"run":item["run"],"seed":item["seed"],"split":split,
                    "utterances":len(rows),"wer_percent":values["wer_percent"],"cer_percent":values["cer_percent"],
                    "macro_speaker_wer_percent":report["macro_speaker_wer_percent"],**values["word"],
                    "raw_wer_percent":report["raw"]["wer_percent"],"generation_cap_count":saved["generation_cap_count"]})
                for field,destination in [("speaker",speaker_rows),("prompt",prompt_rows)]:
                    for key,values in report["by_"+field].items():
                        destination.append({"mode":item["mode"],"run":item["run"],"seed":item["seed"],"split":split,
                            field:key,"utterances":values["utterances"],"wer_percent":values["wer_percent"],
                            "cer_percent":values["cer_percent"],**values["word"]})
                details = {d["utterance_id"]:d for d in report["utterance_errors"]}
                for row in rows:
                    detail = details[row["utterance_id"]]
                    error_rows.append({"mode":item["mode"],"run":item["run"],"seed":item["seed"],"split":split,
                        "utterance_id":row["utterance_id"],"speaker":row["speaker"],"prompt":row["prompt"],
                        "reference":row["reference"],"prediction":row["prediction"],**detail["word"],
                        "character_errors":detail["character"]["errors"],"reference_characters":detail["character"]["reference_units"]})

        mean_rows, comparison_rows, sensitivity_rows = [], [], []
        for split,group in FINAL_PROTOCOL["groups"].items():
            speakers = sorted({row["speaker"] for row in group["records"]})
            prompts = sorted({row["prompt"] for row in group["records"]})
            mean_speaker_errors, mean_speaker_words, point_scores = {}, {}, {}
            for mode in ["pretrained","full","lora"]:
                items = [item for item in FINAL_PROTOCOL["models"] if item["mode"] == mode]
                values = [reports[(item["run"],split)]["normalized"] for item in items]
                point_scores[mode] = statistics.mean(v["wer_percent"] for v in values)
                mean_rows.append({"mode":mode,"split":split,"seed_count":len(items),
                    "mean_wer_percent":point_scores[mode],
                    "sample_sd_wer_pp":statistics.stdev(v["wer_percent"] for v in values) if len(values)>1 else "",
                    "mean_cer_percent":statistics.mean(v["cer_percent"] for v in values),
                    "sample_sd_cer_pp":statistics.stdev(v["cer_percent"] for v in values) if len(values)>1 else "",
                    "mean_substitutions":statistics.mean(v["word"]["substitutions"] for v in values),
                    "mean_deletions":statistics.mean(v["word"]["deletions"] for v in values),
                    "mean_insertions":statistics.mean(v["word"]["insertions"] for v in values),
                    "reference_words":values[0]["word"]["reference_units"]})
                per_speaker = [reports[(item["run"],split)]["by_speaker"] for item in items]
                mean_speaker_errors[mode] = np.array([statistics.mean(r[s]["word"]["errors"] for r in per_speaker) for s in speakers])
                mean_speaker_words[mode] = np.array([per_speaker[0][s]["word"]["reference_units"] for s in speakers])
                assert all(np.array_equal(mean_speaker_words[mode],np.array([r[s]["word"]["reference_units"] for s in speakers])) for r in per_speaker)
            assert all(np.array_equal(mean_speaker_words["pretrained"],v) for v in mean_speaker_words.values())
            # Resample speakers together for every model; never resample utterances as independent speakers.
            rng = np.random.default_rng(FINAL_PROTOCOL["uncertainty"]["seed"])
            samples = rng.integers(0,len(speakers),size=(FINAL_PROTOCOL["uncertainty"]["resamples"],len(speakers)))
            denominators = mean_speaker_words["pretrained"][samples].sum(axis=1)
            for left,right in [("full","lora"),("pretrained","full"),("pretrained","lora")]:
                delta = 100*(mean_speaker_errors[left]-mean_speaker_errors[right])[samples].sum(axis=1)/denominators
                low,high = np.percentile(delta,[2.5,97.5])
                speaker_delta = mean_speaker_errors[left]-mean_speaker_errors[right]
                comparison_rows.append({"split":split,"left":left,"right":right,
                    "mean_wer_difference_pp":point_scores[left]-point_scores[right],"speaker_bootstrap_low_pp":float(low),
                    "speaker_bootstrap_high_pp":float(high),"speakers":len(speakers),"prompts":len(prompts),
                    "speakers_helped_by_right":int((speaker_delta>0).sum()),"speakers_worsened_by_right":int((speaker_delta<0).sum()),
                    "speakers_tied":int((speaker_delta==0).sum()),"resamples":FINAL_PROTOCOL["uncertainty"]["resamples"],
                    "bootstrap_seed":FINAL_PROTOCOL["uncertainty"]["seed"],
                    "scope":"exploratory; conditional on these prompts, split, and trained seeds; no multiplicity correction"})
                for omitted in prompts:
                    scores = {}
                    for mode in [left,right]:
                        items = [item for item in FINAL_PROTOCOL["models"] if item["mode"] == mode]
                        per_seed = []
                        for item in items:
                            report = reports[(item["run"],split)]
                            total,removed = report["normalized"]["word"],report["by_prompt"][omitted]["word"]
                            assert total["reference_units"] > removed["reference_units"]
                            per_seed.append(100*(total["errors"]-removed["errors"])/(total["reference_units"]-removed["reference_units"]))
                        scores[mode] = statistics.mean(per_seed)
                    sensitivity_rows.append({"split":split,"left":left,"right":right,"omitted_prompt":omitted,
                        "wer_difference_pp":scores[left]-scores[right],"scope":"leave-one-prompt-out sensitivity; not a confidence interval"})

        efficiency_rows = []
        for item in FINAL_PROTOCOL["models"]:
            meta = json.loads((ROOT/"runs"/item["run"]/"metadata.json").read_text())
            efficiency_rows.append({"mode":item["mode"],"run":item["run"],"seed":item["seed"],
                "trainable_parameters":meta.get("trainable_parameters",0),
                "recorded_adaptation_end_to_end_minutes":meta["elapsed_seconds"]/60 if item["mode"]!="pretrained" else "",
                "recorded_adaptation_peak_allocated_gib":meta["peak_gpu_allocated_bytes"]/2**30 if item["mode"]!="pretrained" else "",
                "selected_checkpoint_bytes":item["checkpoint_bytes"] if item["mode"]!="pretrained" else "",
                "inference_seconds_447_recordings":sum(json.loads((FINAL_ROOT/item["run"]/split/"metadata.json").read_text())["inference_seconds"] for split in FINAL_PROTOCOL["groups"]),
                "checkpoint_scope":"adapter requires pinned base model" if item["mode"]=="lora" else "full model" if item["mode"]=="full" else "pinned pretrained revision"})
        tables = FINAL_ROOT/"tables"
        tables.mkdir(exist_ok=True)
        for name,rows in [("per_run",run_rows),("model_summary",mean_rows),("speakers",speaker_rows),("prompts",prompt_rows),
                          ("utterance_errors",error_rows),("paired_comparisons",comparison_rows),
                          ("prompt_sensitivity",sensitivity_rows),("efficiency",efficiency_rows)]:
            write_csv(tables/(name+".csv"),rows,list(rows[0]))
        write_json(tables/"sources.json",{"protocol_sha256":FINAL_PROTOCOL_HASH,"files":source_hashes,
            "table_sha256":{p.name:sha256(p) for p in tables.glob("*.csv")},
            "status":FINAL_PROTOCOL["status"],"new_training":False})
        for row in mean_rows:
            print(row["split"],row["mode"],"mean WER:",round(row["mean_wer_percent"],2),"seed SD:",row["sample_sd_wer_pp"])
        print("Positive left-minus-right difference favors the right model.")
        print("Saved comparisons:",tables)
        print("These intervals are exploratory and conditional; they do not establish clinical utility or universal superiority.")
